# 06 · 生成與 KV Cache

每次生成一個字，前面的字其實**被重算了一遍又一遍**——這是 LLM 推論最大的浪費。**KV cache** 把算過的 Key/Value 存起來重複利用，是讓 ChatGPT 能即時回應的關鍵工程。這堂課親手實作它。

## 學習目標

- 理解自回歸生成為什麼**重複計算**
- 親手實作 **KV cache**：只算新 token 的 K/V，舊的存起來重用
- 驗證「有 cache」與「沒 cache」輸出**完全相同**、但 cache **更快**

## 1. 問題：每一步都重算前文

第 05 課的 `generate` 每生成一個字，就把目前整段序列**重新跑一次模型**。但前面那些字的 Key/Value 上一步早就算過了——重算純粹是浪費。序列越長，浪費越誇張。

**KV cache** 的點子：把每一層算過的 K、V 存起來；下一步只需要算**新 token** 的 Q/K/V，再接上快取的舊 K/V 即可。下面是支援 cache 的 GPT：

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class CachedAttention(nn.Module):
    """支援 KV cache 的因果多頭注意力。"""
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head, self.hd = n_head, n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        split = lambda t: t.view(B, T, self.n_head, self.hd).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)
        if cache is not None:                       # 接上之前算過的 K,V
            pk, pv = cache
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)
        new_cache = (k, v)
        Tk = k.size(2)
        att = (q @ k.transpose(-2, -1)) * self.hd ** -0.5
        if T > 1:                                   # 完整輸入（prefill）才需因果遮罩
            mask = torch.tril(torch.ones(T, Tk, device=x.device), diagonal=Tk - T)
            att = att.masked_fill(mask == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), new_cache

class CachedBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd); self.attn = CachedAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x, cache=None):
        a, new = self.attn(self.ln1(x), cache)
        x = x + a
        x = x + self.ff(self.ln2(x))
        return x, new

class MiniGPTKV(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=3, block_size=64):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([CachedBlock(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, caches=None, pos0=0):
        B, T = idx.shape
        pos = torch.arange(pos0, pos0 + T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        new_caches = []
        for blk, c in zip(self.blocks, caches or [None] * len(self.blocks)):
            x, nc = blk(x, c)
            new_caches.append(nc)
        return self.head(self.ln_f(x)), new_caches

    @torch.no_grad()
    def generate_naive(self, idx, n):
        for _ in range(n):                          # 每步都重算整段前文
            logits, _ = self(idx[:, -self.block_size:])
            nxt = logits[:, -1, :].argmax(-1, keepdim=True)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

    @torch.no_grad()
    def generate_cached(self, idx, n):
        logits, caches = self(idx)                  # prefill：算一次 prompt
        out = idx
        nxt = logits[:, -1, :].argmax(-1, keepdim=True)
        for _ in range(n):
            out = torch.cat([out, nxt], dim=1)
            pos0 = out.size(1) - 1                   # 只餵「新的那一個 token」
            logits, caches = self(nxt, caches, pos0=pos0)
            nxt = logits[:, -1, :].argmax(-1, keepdim=True)
        return out

In [2]:
text = """床前明月光，疑是地上霜。舉頭望明月，低頭思故鄉。
春眠不覺曉，處處聞啼鳥。夜來風雨聲，花落知多少。
白日依山盡，黃河入海流。欲窮千里目，更上一層樓。
紅豆生南國，春來發幾枝。願君多采擷，此物最相思。
空山不見人，但聞人語響。返景入深林，復照青苔上。
千山鳥飛絕，萬徑人蹤滅。孤舟蓑笠翁，獨釣寒江雪。
松下問童子，言師采藥去。只在此山中，雲深不知處。
人閒桂花落，夜靜春山空。月出驚山鳥，時鳴春澗中。
君自故鄉來，應知故鄉事。來日綺窗前，寒梅著花未。
獨坐幽篁裡，彈琴復長嘯。深林人不知，明月來相照。
山中相送罷，日暮掩柴扉。春草明年綠，王孫歸不歸。
功蓋三分國，名成八陣圖。江流石不轉，遺恨失吞吳。
前不見古人，後不見來者。念天地之悠悠，獨愴然而涕下。
葡萄美酒夜光杯，欲飲琵琶馬上催。醉臥沙場君莫笑，古來征戰幾人回。
秦時明月漢時關，萬里長征人未還。但使龍城飛將在，不教胡馬度陰山。
朝辭白帝彩雲間，千里江陵一日還。兩岸猿聲啼不住，輕舟已過萬重山。
故人西辭黃鶴樓，煙花三月下揚州。孤帆遠影碧空盡，唯見長江天際流。
月落烏啼霜滿天，江楓漁火對愁眠。姑蘇城外寒山寺，夜半鐘聲到客船。
獨在異鄉為異客，每逢佳節倍思親。遙知兄弟登高處，遍插茱萸少一人。
日照香爐生紫煙，遙看瀑布掛前川。飛流直下三千尺，疑是銀河落九天。
國破山河在，城春草木深。感時花濺淚，恨別鳥驚心。
岐王宅裡尋常見，崔九堂前幾度聞。正是江南好風景，落花時節又逢君。
渭城朝雨浥輕塵，客舍青青柳色新。勸君更盡一杯酒，西出陽關無故人。
清明時節雨紛紛，路上行人欲斷魂。借問酒家何處有，牧童遙指杏花村。
千里黃雲白日曛，北風吹雁雪紛紛。莫愁前路無知己，天下誰人不識君。
"""
import torch

# 字元級詞表：每個不同的字 = 一個 token
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
print(f"語料長度 {len(text)} 字，詞表大小 {vocab_size}")

語料長度 715 字，詞表大小 328


## 2. 驗證：輸出相同、速度更快

用同一個（這裡未特別訓練的）模型，比較 `generate_naive` 與 `generate_cached`。用 **greedy（argmax）** 解碼，兩者應該逐字相同——藉此證明 cache 沒算錯。

In [3]:
import time

torch.manual_seed(0)
# block_size 設大一點，容納 prompt + 生成的總長度（KV cache 用絕對位置）
model = MiniGPTKV(vocab_size, n_embd=128, n_head=4, n_layer=3, block_size=256).eval()
prompt = torch.tensor([[stoi["春"]]])
N = 200

t = time.perf_counter()
a = model.generate_naive(prompt, N)
t_naive = time.perf_counter() - t

t = time.perf_counter()
b = model.generate_cached(prompt, N)
t_cached = time.perf_counter() - t

print("兩種方法輸出完全相同:", torch.equal(a, b))   # cache 正確性檢查
print(f"naive  生成 {N} 字：{t_naive * 1000:7.1f} ms")
print(f"cached 生成 {N} 字：{t_cached * 1000:7.1f} ms　→ 約快 {t_naive / t_cached:.1f}×")

兩種方法輸出完全相同: True
naive  生成 200 字：  413.5 ms
cached 生成 200 字：  137.3 ms　→ 約快 3.0×


輸出一字不差，但 cache 版明顯更快——而且**序列越長，差距越大**（naive 的成本隨長度平方成長，cached 接近線性）。真實的 LLM 服務沒有 KV cache 根本不可能即時回應。

> 代價是**記憶體**：cache 要存每一層、每個 token 的 K、V。這也是為什麼長對話很吃顯示記憶體（VRAM）。

## 小結

- 自回歸生成若每步重算前文，浪費巨大。
- **KV cache** 存下舊 token 的 K/V，每步只算新 token → 大幅加速。
- 正確實作下，輸出與 naive 完全一致；代價是記憶體。

## 練習

1. 把 `N` 從 200 加到 500，加速比變大還是變小？為什麼？
2. 想一想：為什麼長對話會「越聊越吃記憶體」？（提示：cache 隨 token 數成長）

下一課，讓模型從「只會接字」進化到「**會照指令回答**」——對齊的第一步 SFT。